In [1]:
import sys
import os
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
# 문서가 변액일임펀드 설정/해지 지시서인지 확인하는 LLM 노드 생성

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
# LLM 모델 정의

LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")
LLM_TEMPERATURE=os.getenv("LLM_TEMPERATURE")

def create_llm_model():
    # vLLM 모델 인스턴스 생성
    llm = init_chat_model(
        "openai:gpt-4o",
        temperature=LLM_TEMPERATURE,
        top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.9로 설정
    )

    # vLLM 모델 인스턴스 생성
    # llm = init_chat_model(
    #     "openai:",
    #     temperature=LLM_TEMPERATURE,
    #     top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.9로 설정
    #     base_url=LLM_BASE_URL,
    #     api_key=LLM_API_KEY
    # )
    return llm

In [3]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

def chatbot():

    # 메시지 객체 생성
    system_msg = SystemMessage("당신은 자산운용사 업무에 대한 업무 지원을 담당하는 상담원입니다.")
    human_msg = HumanMessage(f"""
아래는 브로커가 보내온 주식 해외거래체결내역 확인서의 메타 데이터입니다.
신입사원과 같이 업무를 처음 접하는 사람이 확인서에서 데이터 추출을 잘 할 수 있도록 각 데이터의 자세한 설명과 일반적인 형식 및 각 단어 또는 코드의 의미를 예시와 함께 자세히 설명하세요.

**markdown table 형식으로 출력하세요.**
**생성한 markdown table이 정상적으로 출력되도록 코드 검수를 진행하세요.**

========================
주식 해외거래체결내역 확인서 데이터 필드
========================
  - Trade Date
  - Fund Code
  - Fund Name/Account Name
  - Ticker
  - ISIN
  - Security Name
  - Settlement Date
  - Buy/Sell
  - Currency
  - Excuted Qty
  - Deal Price
  - Gross Amount
  - Commission
  - Taxes
  - Other Charges
  - Net Settlement AMT
  - Executing Broker
  - Clearing Broker
  - Settlement Location (PSET)
  - Sec Account
  - Clearing Agent ID

    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

In [4]:
from IPython.display import Markdown, display

def display_markdown(response):
    # LLM 응답을 마크다운 형식으로 보기 좋게 표시
    if 'response' in locals():
        display(Markdown(response.content))
        
        # 추가 정보 (토큰 사용량 등)를 표시
        if hasattr(response, 'response_metadata') and response.response_metadata:
            metadata = response.response_metadata
            if 'token_usage' in metadata:
                print("\n---")
                print("**토큰 사용량:**")
                print(f"- 입력 토큰: {metadata['token_usage'].get('prompt_tokens', 'N/A')}")
                print(f"- 출력 토큰: {metadata['token_usage'].get('completion_tokens', 'N/A')}")
                print(f"- 총 토큰: {metadata['token_usage'].get('total_tokens', 'N/A')}")
    else:
        print("⚠️ 'response' 변수를 찾을 수 없습니다. 먼저 LLM을 호출해주세요.")

In [5]:
bot_response = chatbot()
display_markdown(bot_response)

```markdown
| 데이터 필드            | 설명                                                                 | 형식 및 예시                                                                 |
|------------------------|----------------------------------------------------------------------|------------------------------------------------------------------------------|
| Trade Date             | 거래가 체결된 날짜를 의미합니다.                                      | YYYY-MM-DD 형식, 예: 2023-10-15                                              |
| Fund Code              | 펀드를 식별하는 고유 코드입니다.                                      | 알파벳과 숫자의 조합, 예: FND12345                                           |
| Fund Name/Account Name | 펀드 또는 계좌의 이름입니다.                                          | 텍스트 형식, 예: Global Equity Fund                                          |
| Ticker                 | 주식의 티커 심볼로, 주식을 식별하는 코드입니다.                       | 알파벳 조합, 예: AAPL (Apple Inc.)                                           |
| ISIN                   | 국제 증권 식별 번호로, 증권을 식별하는 국제 표준 코드입니다.           | 알파벳과 숫자의 조합, 예: US0378331005                                       |
| Security Name          | 증권의 정식 명칭입니다.                                               | 텍스트 형식, 예: Apple Inc.                                                  |
| Settlement Date        | 거래가 실제로 결제되는 날짜입니다.                                    | YYYY-MM-DD 형식, 예: 2023-10-18                                              |
| Buy/Sell               | 매수 또는 매도를 나타냅니다.                                          | "Buy" 또는 "Sell"                                                            |
| Currency               | 거래에 사용된 통화입니다.                                             | 통화 코드, 예: USD (미국 달러)                                               |
| Executed Qty           | 체결된 주식의 수량입니다.                                             | 숫자 형식, 예: 1000                                                          |
| Deal Price             | 주식의 체결 가격입니다.                                               | 소수점 포함 숫자, 예: 150.25                                                 |
| Gross Amount           | 총 거래 금액으로, 수량과 가격의 곱입니다.                             | 소수점 포함 숫자, 예: 150250.00                                              |
| Commission             | 거래 수수료입니다.                                                    | 소수점 포함 숫자, 예: 150.25                                                 |
| Taxes                  | 거래에 부과된 세금입니다.                                             | 소수점 포함 숫자, 예: 15.00                                                  |
| Other Charges          | 기타 부대 비용입니다.                                                 | 소수점 포함 숫자, 예: 5.00                                                   |
| Net Settlement AMT     | 최종 결제 금액으로, 총 금액에서 수수료, 세금, 기타 비용을 뺀 금액입니다.| 소수점 포함 숫자, 예: 150079.75                                              |
| Executing Broker       | 거래를 실행한 브로커의 이름입니다.                                    | 텍스트 형식, 예: ABC Securities                                              |
| Clearing Broker        | 거래를 청산한 브로커의 이름입니다.                                    | 텍스트 형식, 예: XYZ Clearing                                                |
| Settlement Location (PSET) | 결제가 이루어지는 장소를 나타냅니다.                                | 텍스트 형식, 예: New York                                                    |
| Sec Account            | 증권 계좌 번호입니다.                                                 | 숫자 형식, 예: 123456789                                                     |
| Clearing Agent ID      | 청산 기관의 식별 번호입니다.                                          | 알파벳과 숫자의 조합, 예: CL123456                                           |
```



---
**토큰 사용량:**
- 입력 토큰: 276
- 출력 토큰: 727
- 총 토큰: 1003
